# H65 Stage 0 — Colab CUDA

Chọn **Kernel → Colab → Auto Connect** trước khi chạy. Notebook này chỉ benchmark Stage 0 và sẽ dừng nếu CUDA/gate không đạt; chưa tạo output hoặc ZIP. Repository đã public nên notebook không yêu cầu GitHub token.

In [23]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/ViettelAIRACE')
assert (DRIVE_ROOT / 'turn2/input').exists(), 'Đặt turn2/input vào MyDrive/ViettelAIRACE trước'
print('Drive root:', DRIVE_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive root: /content/drive/MyDrive/ViettelAIRACE


In [24]:
import os
import shutil
import subprocess

os.chdir('/content')
repo_dir = Path('/content/ViettelAIRACE')
if repo_dir.exists():
    shutil.rmtree(repo_dir)

clone_result = subprocess.run(
    [
        'git', 'clone', '--branch', 'codex/core-rebuild-h57', '--single-branch',
        'https://github.com/minggu151623/ViettelAIRACE.git', str(repo_dir),
    ],
    text=True, capture_output=True,
)
if clone_result.returncode != 0:
    print(clone_result.stderr)
    raise RuntimeError('Không clone được repository public; kiểm tra mạng hoặc trạng thái GitHub.')

os.chdir(repo_dir)
(repo_dir / 'turn2').mkdir(parents=True, exist_ok=True)
(repo_dir / 'experiments/H65_crosslingual_projection_corrected_execution/results').mkdir(parents=True, exist_ok=True)
(repo_dir / 'external').mkdir(parents=True, exist_ok=True)
target_input = repo_dir / 'turn2/input'
if target_input.exists():
    shutil.rmtree(target_input)
shutil.copytree(DRIVE_ROOT / 'turn2/input', target_input)
input_files = sorted(target_input.glob('*.txt'))
assert len(input_files) == 100, f'Expected 100 input files, found {len(input_files)}'
subprocess.run(
    ['git', 'clone', '--depth', '1', 'https://github.com/VinAIResearch/PhoNER_COVID19.git', 'external/PhoNER_COVID19'],
    check=True,
)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Repository ready:', repo_dir, commit)

Cloning into '/content/ViettelAIRACE'...
fatal: Unable to read current working directory: No such file or directory



RuntimeError: Không clone được repository public; kiểm tra mạng hoặc trạng thái GitHub.

In [ ]:
!pip -q install -U transformers sentencepiece accelerate huggingface_hub
import torch
assert torch.cuda.is_available(), 'CUDA chưa bật: Runtime → Change runtime type → GPU'
print(torch.cuda.get_device_name(0), torch.__version__)

In [ ]:
import subprocess
stage0_result = subprocess.run([
    'python', 'experiments/H65_crosslingual_projection_corrected_execution/run_stage_0_colab.py',
    '--phoner-dev', 'external/PhoNER_COVID19/data/word/dev_word.json',
    '--input', 'turn2/input',
    '--output', 'experiments/H65_crosslingual_projection_corrected_execution/results',
])
stage0_report = Path('experiments/H65_crosslingual_projection_corrected_execution/results/stage_0_report.json')
if stage0_result.returncode != 0 and not stage0_report.exists():
    raise RuntimeError(f'Stage-0 crashed before writing a report (exit {stage0_result.returncode}); inspect the traceback above.')
print('Stage-0 exit code:', stage0_result.returncode, '(report will be shown in the next cell)')

In [ ]:
from pathlib import Path
import json
report_path = Path('experiments/H65_crosslingual_projection_corrected_execution/results/stage_0_report.json')
assert report_path.exists(), 'Stage-0 không tạo report; hãy xem traceback ở cell 4'
report = json.loads(report_path.read_text())
display(report)
print('Copy this report back to Codex. Do not run Turn-2 if status is FAIL.')